# Лабораторная работа «Свёрточные нейронные сети»
## Часть 3. Zero-shot классификация с помощью CLIP

CLIP отображает изображения и тексты в общее векторное пространство. Чтобы классифицировать картинку без обучения, для каждого класса составляется текстовый промпт, и выбирается тот класс, чей текстовый вектор ближе всего (по косинусной близости) к вектору изображения.

Проверяем на тестовой части Oxford Pets (та же выборка, что в части 2): бинарная классификация кошка/собака и классификация по 37 породам. Сеть нигде не дообучается.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.datasets import OxfordIIITPet
from transformers import CLIPModel, CLIPTokenizer

from sklearn.metrics import confusion_matrix

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Устройство:', device)

### Данные

In [ ]:
OxfordIIITPet('data', download=True)   # скачивание и распаковка (около 800 МБ)
ROOT = 'data/oxford-iiit-pet'


def read_split(name):
    rows = []
    with open(os.path.join(ROOT, 'annotations', name + '.txt')) as f:
        for line in f:
            if not line.strip() or line.startswith('#'):
                continue
            img, cls, species = line.split()[:3]
            rows.append((os.path.join(ROOT, 'images', img + '.jpg'), int(cls) - 1, int(species) - 1))
    return rows   # (путь, порода, вид: 0 - кошка, 1 - собака)


rows = read_split('test')
NUM_CLASSES = len(set(r[1] for r in rows))

classes = [''] * NUM_CLASSES
class_is_dog = np.zeros(NUM_CLASSES, dtype=bool)
for path, c, s in rows:
    classes[c] = os.path.basename(path).rsplit('_', 1)[0]
    class_is_dog[c] = (s == 1)

y_true = torch.tensor([r[1] for r in rows])
true_dog = torch.tensor([r[2] == 1 for r in rows])

print(f'изображений: {len(rows)} | классов: {NUM_CLASSES}')
print('кошек:', (~true_dog).sum().item(), '| собак:', true_dog.sum().item())

### Модель

Используются оригинальные веса OpenAI CLIP (ViT-B/16), загруженные через библиотеку `transformers`. Предобработка изображений та же, что у авторов CLIP.

In [ ]:
MODEL = 'openai/clip-vit-base-patch16'
model = CLIPModel.from_pretrained(MODEL).eval().to(device)
tokenizer = CLIPTokenizer.from_pretrained(MODEL)

clip_tf = T.Compose([
    T.Resize(224, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize([0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711]),
])


class Pets(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        return clip_tf(Image.open(self.rows[i][0]).convert('RGB'))


@torch.no_grad()
def embed_images(rows):
    dl = DataLoader(Pets(rows), batch_size=64, shuffle=False, num_workers=2)
    out = []
    for x in tqdm(dl, desc='изображения'):
        e = model.visual_projection(model.vision_model(pixel_values=x.to(device)).pooler_output)
        out.append(F.normalize(e.float(), dim=-1).cpu())
    return torch.cat(out)


@torch.no_grad()
def embed_texts(prompts):
    tok = tokenizer(prompts, padding=True, return_tensors='pt').to(device)
    e = model.text_projection(model.text_model(input_ids=tok['input_ids'],
                                               attention_mask=tok['attention_mask']).pooler_output)
    return F.normalize(e.float(), dim=-1).cpu()

In [ ]:
# Векторы изображений считаем один раз, дальше меняем только тексты
img_emb = embed_images(rows)
print(tuple(img_emb.shape))

## Кошка или собака

Два промпта: `a photo of a cat` и `a photo of a dog`.

In [ ]:
bin_prompts = ['a photo of a cat', 'a photo of a dog']
bin_sim = img_emb @ embed_texts(bin_prompts).T
pred_dog = bin_sim.argmax(1) == 1
acc_bin = (pred_dog == true_dog).float().mean().item()
print(f'Accuracy кошки/собаки: {acc_bin:.4f}')

pd.DataFrame(confusion_matrix(true_dog.numpy(), pred_dog.numpy(), labels=[False, True]),
             index=['истина: кошка', 'истина: собака'],
             columns=['предсказано: кошка', 'предсказано: собака'])

## Породы

Сравним три варианта промпта: только название породы, `a photo of a ...` и шаблон из статьи CLIP для этого датасета `a photo of a ..., a type of pet`. Для матрицы ошибок используется последний вариант.

In [ ]:
breed_names = [c.replace('_', ' ').lower() for c in classes]
templates = ['{}', 'a photo of a {}', 'a photo of a {}, a type of pet']

results = {}
for t in templates:
    sim = img_emb @ embed_texts([t.format(b) for b in breed_names]).T
    results[t] = sim
    print(f'{t:<35s} accuracy = {(sim.argmax(1) == y_true).float().mean().item():.4f}')

main_template = templates[-1]
sim = results[main_template]
pred = sim.argmax(1)
acc = (pred == y_true).float().mean().item()

# кошка/собака через предсказанную породу
acc_bin_breed = (torch.tensor(class_is_dog)[pred] == true_dog).float().mean().item()

print()
print(f'Accuracy по породам:                         {acc:.4f}')
print(f'Accuracy кошки/собаки (по предсказанной породе): {acc_bin_breed:.4f}')

In [ ]:
n = NUM_CLASSES
cm = confusion_matrix(y_true.numpy(), pred.numpy(), labels=list(range(n)))

fig, ax = plt.subplots(figsize=(14, 12))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(n))
ax.set_xticklabels(classes, rotation=90, fontsize=7)
ax.set_yticks(range(n))
ax.set_yticklabels(classes, fontsize=7)
ax.set_xlabel('Предсказанный класс')
ax.set_ylabel('Истинный класс')
ax.set_title('Confusion matrix: CLIP zero-shot, породы')
for i in range(n):
    for j in range(n):
        if cm[i, j]:
            ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=6,
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
fig.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.show()

In [ ]:
# Какие породы CLIP путает чаще всего
off = cm.copy()
np.fill_diagonal(off, 0)
for k in np.argsort(off.ravel())[::-1][:10]:
    i, j = divmod(int(k), n)
    print(f'{classes[i]:>28s} -> {classes[j]:<28s} {off[i, j]}')

# Точность по каждой породе
per_class = pd.Series(cm.diagonal() / cm.sum(1), index=classes).sort_values()
per_class.plot.barh(figsize=(7, 9), title='Accuracy по породам (CLIP zero-shot)')
plt.tight_layout()
plt.show()

## Итоги

In [ ]:
summary = [('Кошки/собаки, прямые промпты', acc_bin),
           ('Кошки/собаки, через предсказанную породу', acc_bin_breed)]
summary += [('Породы, промпт: ' + t, (results[t].argmax(1) == y_true).float().mean().item()) for t in templates]
pd.DataFrame([(m, round(v, 4)) for m, v in summary], columns=['метрика', 'значение'])